# State Space Models — from Vanilla SSM to Mamba

This notebook is a **self-contained, step-by-step walkthrough** of the `ssm-demo` project.  
Every implementation is defined inline — no local package install required.

---

## Why State Space Models?

Modern sequence modelling has long been dominated by the **Transformer**, which uses
self-attention to relate every token to every other token.
That power comes at a steep cost: attention scales as **O(L²)** in both time and
memory, where *L* is the sequence length.

**State Space Models (SSMs)** offer a fundamentally different approach rooted in
classical control theory.  Instead of comparing all pairs of tokens, an SSM maintains
a compact *hidden state* that is updated one step at a time — giving **O(L)**
complexity.

The progression we implement here is:

| Component | What it adds |
|-----------|-------------|
| **SSM** (vanilla) | Linear recurrence with fixed parameters |
| **SelectiveSSM (S6)** | Parameters become *input-dependent* — the model can choose what to remember |
| **MambaBlock** | S6 wrapped with gating, causal conv, and a residual connection |
| **MambaModel** | Stacked blocks + token embedding → a full language model |

Each layer builds directly on the previous one.  By the end of the notebook you will
have a working Mamba language model trained on toy data.

---

**Contents**
1. [Setup & imports](#1-setup--imports)
2. [Vanilla SSM](#2-vanilla-ssm)
3. [Selective SSM (S6)](#3-selective-ssm-s6)
4. [Mamba Block](#4-mamba-block)
5. [Full Mamba Language Model](#5-full-mamba-language-model)
6. [Demos](#6-demos)
7. [Mini Training Loop](#7-mini-training-loop)


---
## 1. Setup & imports

We need three libraries:

* **PyTorch** — the deep-learning backbone; all tensor operations and `nn.Module` layers.
* **NumPy** — numerical utilities (torch interoperates with it seamlessly).
* **einops** — expressive tensor-reshaping helpers (used in more advanced SSM
  implementations; included here for parity with the package requirements).

Run the cell below once; subsequent cells will use the cached installation.


In [ ]:
%pip install --quiet torch numpy einops


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch {torch.__version__}")
print(f"Device : {'cuda' if torch.cuda.is_available() else 'cpu'}")


---

> **Tensor shape convention used throughout this notebook**
>
> | Symbol | Meaning |
> |--------|---------|
> | `B` | batch size |
> | `L` | sequence length |
> | `d_model` | model (input/output) feature dimension |
> | `d_state` (N) | hidden-state dimension inside the SSM |
> | `d_inner` | expanded inner dimension inside a Mamba block (= `expand × d_model`) |
>
> All sequence tensors are `(B, L, d_model)` — the standard PyTorch convention.


---
## 2. Vanilla SSM

### Background — from differential equations to a learnable layer

A continuous-time linear SSM is defined by two equations:

$$h'(t) = A\,h(t) + B\,u(t) \qquad\qquad y(t) = C\,h(t) + D\,u(t)$$

Think of $h(t)$ as the system’s “memory” of everything it has seen so far.
$A$ drives how that memory evolves on its own; $B$ injects the new input $u(t)$;
$C$ reads out what the memory “says”; and $D$ is a direct skip connection.

### Discretisation with Zero-Order-Hold (ZOH)

Computers work with sequences, not continuous time.  We discretise with a learnable
step size $\Delta$ using the **Zero-Order-Hold** rule:

$$\bar{A} = \exp(\Delta A) \qquad \bar{B} = (\bar{A} - I)\,A^{-1}\,B$$

This gives the **recurrence**:

$$h_t = \bar{A}\,h_{t-1} + \bar{B}\,u_t \qquad y_t = C\,h_t + D\,u_t$$

### Design choices in this implementation

| Choice | Why |
|--------|-----|
| **Diagonal $A$** | A full $d_\text{state} \times d_\text{state}$ matrix would be expensive; the diagonal S4D parameterisation keeps it at $O(d_\text{state})$ parameters per feature. |
| **$A$ stored as $\log\lvert A\rvert$** | Forces $A$ to stay negative (via $A \leftarrow -\exp(A_\text{log})$), which guarantees $\lvert\bar{A}\rvert \le 1$ and a **stable** system — states don’t blow up. |
| **Log-uniform init for $\Delta$** | Spreads time-scales evenly in log-space so the model can represent both fast (small $\Delta$) and slow (large $\Delta$) dynamics. |

### Limitation that Mamba will fix

Here, $A$, $B$, $C$, and $\Delta$ are **fixed** for all input positions — the
recurrence is the same no matter what token is at position $t$.  That makes it
hard to “ignore” irrelevant tokens.  Section 3 addresses this.


In [ ]:
class SSM(nn.Module):
    """Learnable discrete-time diagonal SSM.

    This is the baseline building block.  It is a *linear* recurrent layer:
    the same A, B, C, D matrices are applied at every time step regardless of
    the input content -- making it efficient but not content-selective.

    Parameters
    ----------
    d_model : int
        Input/output feature dimension.
    d_state : int
        Dimension of the hidden state h_t  (called N in S4 literature).
        Larger d_state = richer memory, but more compute per step.
    dt_min, dt_max : float
        Log-uniform range for the learnable time-step Delta.
    """

    def __init__(
        self,
        d_model: int,
        d_state: int = 16,
        dt_min: float = 1e-3,
        dt_max: float = 0.1,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # -- A matrix -----------------------------------------------------
        # Stored as log|A| so that the true A = -exp(A_log) is always
        # negative, keeping |A_bar| <= 1 (stable system).
        # Initialised to -[1, 2, ..., d_state] (HiPPO-inspired spacing).
        # Shape: (d_model, d_state) -- one independent state per input feature.
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0)
        A = A.expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))

        # -- B and C ------------------------------------------------------
        # B: projects input u_t onto the hidden state space.
        # C: projects the hidden state h_t back to the output space.
        # Both are (d_model, d_state) -- shared across ALL time steps.
        self.B = nn.Parameter(torch.randn(d_model, d_state) * 0.01)
        self.C = nn.Parameter(torch.randn(d_model, d_state) * 0.01)

        # -- D (skip connection) ------------------------------------------
        # Adds u_t directly to y_t, like a bypass residual.
        # Initialised to 1 so the output starts close to the input.
        self.D = nn.Parameter(torch.ones(d_model))

        # -- Delta (time step) --------------------------------------------
        # One learnable time-step per input feature, log-uniform in
        # [dt_min, dt_max].  Small Delta => state barely moves (remembers);
        # Large Delta => state tracks the input closely (forgets quickly).
        log_dt = (
            torch.rand(d_model) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        )
        self.log_dt = nn.Parameter(log_dt)

    def _get_AB_bar(self) -> tuple[torch.Tensor, torch.Tensor]:
        """Discretise A and B using the Zero-Order-Hold rule.

        Returns A_bar = exp(Delta * A) and B_bar = (A_bar - I)/A * B,
        both of shape (d_model, d_state).
        These are the matrices actually used in the recurrence.
        """
        dt = torch.exp(self.log_dt)       # (d_model,) -- positive step sizes
        A  = -torch.exp(self.A_log)       # (d_model, d_state) -- negative eigenvalues

        # A_bar = exp(Delta * A): broadcast dt (d_model,) over A (d_model, d_state)
        A_bar = torch.exp(dt.unsqueeze(-1) * A)

        # B_bar = (A_bar - I) / A * B  -- exact ZOH formula for diagonal A
        B_bar = (A_bar - 1.0) / A * self.B

        return A_bar, B_bar

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        """Run the SSM recurrence over every time step.

        The loop is straightforward:
          for t in 0..L-1:
              h_t = A_bar * h_{t-1}  +  B_bar * u_t   (update hidden state)
              y_t = C * h_t          +  D     * u_t   (read out + skip)

        In the larger Mamba architecture this module is replaced by
        SelectiveSSM where A_bar and B_bar change at every step based
        on the input.

        Parameters
        ----------
        u : (B, L, d_model)

        Returns
        -------
        y : (B, L, d_model)  -- same shape as input
        """
        B_batch, L, d = u.shape
        assert d == self.d_model, f"Expected d_model={self.d_model}, got {d}"

        A_bar, B_bar = self._get_AB_bar()  # (d_model, d_state) -- fixed for all t
        C = self.C                          # (d_model, d_state)
        D = self.D                          # (d_model,)

        # Hidden state: one vector per (batch item, feature)
        h = torch.zeros(B_batch, d, self.d_state, device=u.device, dtype=u.dtype)

        ys = []
        for t in range(L):
            u_t = u[:, t, :]                # (B, d_model) -- current input slice
            # State update (element-wise multiply because A is diagonal)
            h = A_bar.unsqueeze(0) * h + B_bar.unsqueeze(0) * u_t.unsqueeze(-1)
            # Output: sum over the state dimension, then add skip connection
            y_t = (h * C.unsqueeze(0)).sum(-1) + D * u_t
            ys.append(y_t)

        return torch.stack(ys, dim=1)       # (B, L, d_model)


---
## 3. Selective SSM (S6)

### The key idea: content-aware recurrence

The vanilla SSM uses the same $\bar{A}$ and $\bar{B}$ for *every* time step,
regardless of what the input token actually is.  This is analogous to an RNN with
fixed weights — it can model sequences, but it cannot decide at position $t$ whether
that token is worth remembering.

Mamba’s central contribution is to make $\Delta$, $B$, and $C$ **functions of the
input** at each step:

$$\Delta_t,\;B_t,\;C_t = f(x_t)$$

Because $\Delta$ controls how much the state is updated (small $\Delta$
$\Rightarrow$ state barely changes; large $\Delta$ $\Rightarrow$ state tracks the
input closely), the model can effectively **gate information in and out** — similar
in spirit to LSTM gates, but implemented through the SSM formalism.

### Why not make $A$ input-dependent too?

$A$ governs the long-range decay structure of the model.  Making it input-dependent
would break the ability to use efficient parallel scans (a hardware optimisation used
in the full Mamba paper).  Keeping $A$ fixed also improves stability.

### Projection scheme

The three selective parameters are produced by a single linear projection:

$$[\Delta_\text{raw},\; B,\; C] = x\, W_\text{proj}
\qquad (\text{shape: } B \times L \times (dt\_rank + 2\cdot d\_state))$$

$\Delta_\text{raw}$ is then up-projected and passed through **softplus** to ensure
positivity:

$$\Delta_t = \text{softplus}(W_{dt}\,\Delta_\text{raw,t} + b_{dt})$$

The bias is carefully initialised so initial $\Delta$ values fall in $[dt\_min,\ dt\_max]$.

### Selective scan recurrence

$$\bar{A}_t = \exp(\Delta_t \cdot A), \quad \bar{B}_t = \Delta_t \cdot B_t$$
$$h_t = \bar{A}_t \cdot h_{t-1} + \bar{B}_t \cdot x_t, \quad y_t = C_t \cdot h_t + D \cdot x_t$$

### Where S6 fits in the bigger picture

`SelectiveSSM` is **not** used directly by end-users.  It is a sub-module inside
`MambaBlock`, which adds gating, a causal convolution, and a residual connection
around it.  Think of it as the “core engine” that all the surrounding machinery
is built to serve.


In [ ]:
class SelectiveSSM(nn.Module):
    """S6: the selective state-space scan at the heart of every Mamba block.

    Unlike the vanilla SSM, the recurrence matrices change at every time step
    because Delta, B, and C are projected from the current input.

    Parameters
    ----------
    d_inner : int
        Inner channel dimension (= expand * d_model, set by MambaBlock).
    d_state : int
        SSM state dimension N.  Larger => richer memory, slower scan.
    dt_rank : int | None
        Rank of the Delta projection.  Mamba paper uses ceil(d_model / 16)
        as a low-rank bottleneck: dt_rank << d_inner keeps the projection cheap.
    dt_min, dt_max : float
        Log-uniform range for the Delta bias initialisation.
    """

    def __init__(
        self,
        d_inner: int,
        d_state: int = 16,
        dt_rank: int | None = None,
        dt_min: float = 1e-3,
        dt_max: float = 0.1,
    ) -> None:
        super().__init__()
        self.d_inner = d_inner
        self.d_state = d_state
        # Low-rank bottleneck for Delta: avoids a large d_inner -> d_inner projection
        self.dt_rank = dt_rank if dt_rank is not None else math.ceil(d_inner / 16)

        # -- A: fixed structure, learned as log|A| -----------------------
        # Same stability trick as in the vanilla SSM.
        # Not made input-dependent: keeps the long-range decay structure
        # stable and compatible with efficient parallel-scan implementations.
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0)
        A = A.expand(d_inner, -1)           # (d_inner, d_state)
        self.A_log = nn.Parameter(torch.log(A))

        # -- D: global skip connection ------------------------------------
        self.D = nn.Parameter(torch.ones(d_inner))

        # -- x_proj: joint projection that produces Delta_raw, B, and C --
        # Output: (dt_rank + 2*d_state) per position.
        # One projection is more efficient than three separate ones.
        self.x_proj = nn.Linear(d_inner, self.dt_rank + 2 * d_state, bias=False)

        # -- dt_proj: up-project Delta_raw (dt_rank) -> Delta (d_inner) --
        # The bias is initialised so that softplus(bias) ~ U[dt_min, dt_max].
        self.dt_proj = nn.Linear(self.dt_rank, d_inner, bias=True)
        dt_init = (
            torch.rand(d_inner) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        )
        dt_init = torch.exp(dt_init)
        with torch.no_grad():
            # inverse softplus: softplus(x) = y  =>  x = log(exp(y) - 1)
            self.dt_proj.bias.copy_(torch.log(torch.expm1(dt_init)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the selective scan over all L positions.

        Key difference from SSM.forward:
          * dA and dB are (B, L, d_inner, d_state) -- they vary per position.
          * The scan loop therefore uses dA[:, t] and dB[:, t] at each step.

        Parameters
        ----------
        x : (B, L, d_inner)   -- comes from the conv branch of MambaBlock

        Returns
        -------
        y : (B, L, d_inner)
        """
        B_batch, L, _ = x.shape

        # A stays negative -- real-part stability guaranteed
        A = -torch.exp(self.A_log.float())  # (d_inner, d_state)

        # -- Compute input-dependent Delta, B, C in one pass -------------
        x_dbl = self.x_proj(x)              # (B, L, dt_rank + 2*d_state)
        dt_raw, B_mat, C_mat = x_dbl.split(
            [self.dt_rank, self.d_state, self.d_state], dim=-1
        )
        # Softplus keeps Delta positive; the bias shifts the initial distribution
        dt = F.softplus(self.dt_proj(dt_raw))  # (B, L, d_inner)

        # -- Discretise A and B for every position -----------------------
        # dA[b, t, i, n] = exp( dt[b,t,i] * A[i,n] )
        dA = torch.exp(dt.unsqueeze(-1) * A)          # (B, L, d_inner, d_state)
        # dB[b, t, i, n] = dt[b,t,i] * B_mat[b,t,n]  (Euler approximation for B)
        dB = dt.unsqueeze(-1) * B_mat.unsqueeze(2)    # (B, L, d_inner, d_state)

        # -- Sequential selective scan ------------------------------------
        # h: running hidden state, (B, d_inner, d_state)
        h = x.new_zeros(B_batch, self.d_inner, self.d_state)
        ys = []
        for t in range(L):
            # State update -- dA and dB are now time-varying
            h = dA[:, t] * h + dB[:, t] * x[:, t].unsqueeze(-1)
            # Read out: contract over state dimension using time-varying C
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(-1)  # (B, d_inner)
            ys.append(y_t)

        y = torch.stack(ys, dim=1)          # (B, L, d_inner)
        y = y + x * self.D                  # skip connection
        return y


---
## 4. Mamba Block

### What MambaBlock adds around SelectiveSSM

`SelectiveSSM` is a powerful recurrence, but on its own it is missing several
ingredients that make a practical deep-learning layer:

1. **Input normalisation** — `LayerNorm` before the projections keeps activations
   well-conditioned during training, especially in deep stacks.
2. **Causal convolution** — a short depthwise `Conv1d` (kernel size `d_conv`)
   applied *before* the SSM gives the model local context (a sliding window of
   the past `d_conv` tokens) without violating causality.  Padding `d_conv − 1`
   on the left ensures position $t$ only ever sees tokens $\le t$.
3. **SiLU gating** — a second branch computes a gate $z$ that is multiplied
   element-wise with the SSM output.  This lets the block suppress SSM outputs
   that are irrelevant, similar to how GRU/LSTM gates work.
4. **Residual connection** — the block’s output is added back to the original
   input, making deep stacks easier to train (same motivation as ResNets and
   Transformer residuals).

### Block diagram

```
x : (B, L, d_model)
|
|   LayerNorm
|   in_proj  ->  2 x d_inner
|       |-- x_ssm  ->  Conv1d (causal, depthwise)  ->  SiLU  ->  SelectiveSSM --|
|       |-- z      ->  SiLU  --------------------------------------------------|  x  ->  out_proj  ->  d_model
|
|-- + residual
```

### Role in MambaModel

`MambaBlock` is the **repeating unit** of the full model — `MambaModel` is simply
$N$ of these blocks stacked in sequence.  Each block refines the hidden
representation, letting the model build up progressively more abstract features
across layers, exactly as Transformer blocks do.


In [ ]:
class MambaBlock(nn.Module):
    """A single Mamba residual block.

    Composes a SelectiveSSM with normalisation, a causal depthwise convolution,
    SiLU gating, and a residual skip.  This is the repeating unit stacked N
    times in MambaModel.

    Parameters
    ----------
    d_model : int
        Model (input/output) dimension -- unchanged by the block.
    d_state : int
        SSM state dimension passed to SelectiveSSM.
    d_conv : int
        Kernel size of the depthwise causal convolution.  Typical value: 4.
        Larger => wider local context window, more parameters.
    expand : int
        Channel expansion factor.  d_inner = expand * d_model.
        Expanding to 2x (default) before the SSM improves representational
        capacity without changing the interface dimension.
    """

    def __init__(
        self,
        d_model: int,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_inner = expand * d_model   # wider internal representation

        # -- Input normalisation ------------------------------------------
        # Applied before the projections (pre-norm style, like modern Transformers).
        # Keeps gradients stable in deep stacks.
        self.norm = nn.LayerNorm(d_model)

        # -- Input projection: d_model -> 2 * d_inner ---------------------
        # Produces both branches (SSM + gate) in a single matmul.
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)

        # -- Causal depthwise Conv1d --------------------------------------
        # groups=d_inner: each channel convolved independently (depthwise).
        # padding=d_conv-1 on the left: gives exactly d_conv-1 history tokens
        # at position t=0, so no future token ever leaks in (causal).
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=d_conv,
            groups=self.d_inner,    # depthwise
            padding=d_conv - 1,     # causal left-padding
            bias=True,
        )

        # -- Selective SSM ------------------------------------------------
        # The core recurrence: receives the convolved, activated representation.
        self.ssm = SelectiveSSM(d_inner=self.d_inner, d_state=d_state)

        # -- Output projection: d_inner -> d_model -----------------------
        # Brings the gated SSM output back to the residual stream dimension.
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the Mamba block with a residual connection.

        The two-branch structure mirrors a gated linear unit (GLU):
          * Branch 1 (SSM):  conv -> SiLU -> SelectiveSSM
          * Branch 2 (gate): SiLU
          * Output:          out_proj(branch1 * branch2) + residual

        Parameters
        ----------
        x : (B, L, d_model)

        Returns
        -------
        out : (B, L, d_model)  -- same shape as input
        """
        residual = x                         # saved for the residual addition

        # Normalise before projecting
        x = self.norm(x)

        # Project to 2 * d_inner and split into the two branches
        xz = self.in_proj(x)                 # (B, L, 2*d_inner)
        x_ssm, z = xz.chunk(2, dim=-1)       # each (B, L, d_inner)

        # -- SSM branch --------------------------------------------------
        # Step 1: causal depthwise conv -- injects local context
        x_ssm = x_ssm.transpose(1, 2)        # Conv1d expects (B, C, L)
        x_ssm = self.conv1d(x_ssm)[:, :, : x.shape[1]]  # trim left-padding
        x_ssm = x_ssm.transpose(1, 2)        # back to (B, L, d_inner)
        # Step 2: non-linearity before the SSM
        x_ssm = F.silu(x_ssm)
        # Step 3: selective SSM recurrence
        x_ssm = self.ssm(x_ssm)              # (B, L, d_inner)

        # -- Gate branch -------------------------------------------------
        # z acts as a learned gate: values near 0 suppress the SSM output;
        # large magnitudes let it through.
        z = F.silu(z)                         # (B, L, d_inner)

        # -- Combine -----------------------------------------------------
        out = self.out_proj(x_ssm * z)        # element-wise gate then project
        return out + residual                 # residual connection


---
## 5. Full Mamba Language Model

### From a single block to a language model

`MambaModel` assembles the full pipeline:

```
Token IDs (B, L)
    |  nn.Embedding           -- look up a d_model-dimensional vector per token
    v
(B, L, d_model)
    |  MambaBlock x N         -- refine representations layer by layer
    v
(B, L, d_model)
    |  LayerNorm               -- stabilise before the final projection
    v
(B, L, d_model)
    |  nn.Linear (lm_head)    -- project to vocabulary logits
    v
(B, L, vocab_size)  -- unnormalised log-probabilities over next tokens
```

### Why stack multiple blocks?

A single MambaBlock can only capture patterns accessible in one pass of the
selective scan.  Stacking $N$ blocks lets the model build **hierarchical
representations**: early layers learn low-level token patterns; later layers
combine them into higher-level structure — the same intuition as deep CNNs or
Transformers.

### Weight tying

The `lm_head` output matrix is set to the *same tensor* as `embedding.weight`:

```python
self.lm_head.weight = self.embedding.weight
```

This classic trick (Press & Wolf, 2017):
* **Reduces parameters** — `vocab_size × d_model` floats are shared, not doubled.
* **Improves generalisation** — the model is forced to use semantically similar
  representations for encoding and decoding the same token.

### Causal language modelling objective

At inference time the model autoregressively predicts the next token given all
previous ones.  During training we use **teacher forcing**: feed the ground-truth
prefix and compute cross-entropy against the next-token targets in parallel.
This is exactly what the training loop in Section 7 demonstrates.


In [ ]:
class MambaModel(nn.Module):
    """Causal sequence model built from stacked Mamba blocks.

    This is the full end-to-end model: token IDs in, next-token logits out.
    It can be trained for language modelling or any other sequence prediction task.

    Parameters
    ----------
    vocab_size : int
        Number of distinct token types (vocabulary size).
    d_model : int
        Dimensionality of token embeddings and residual stream.
    n_layers : int
        Number of MambaBlock layers to stack.  More layers => more capacity
        but slower training and higher memory usage.
    d_state : int
        SSM state dimension passed to each MambaBlock (and hence SelectiveSSM).
    d_conv : int
        Causal conv kernel size for each block.
    expand : int
        Inner expansion factor for each block.
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_layers: int = 4,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
    ) -> None:
        super().__init__()

        # -- Token embedding ----------------------------------------------
        # Converts integer token IDs -> d_model-dimensional vectors.
        # These vectors are the 'residual stream' that blocks read from and write to.
        self.embedding = nn.Embedding(vocab_size, d_model)

        # -- Stack of Mamba blocks ----------------------------------------
        # Each block is identical in architecture; they differ only in their
        # learned weights.  ModuleList registers them as sub-modules so
        # optimizer.parameters() and .to(device) work correctly.
        self.layers = nn.ModuleList(
            [
                MambaBlock(
                    d_model=d_model,
                    d_state=d_state,
                    d_conv=d_conv,
                    expand=expand,
                )
                for _ in range(n_layers)
            ]
        )

        # -- Final layer norm ---------------------------------------------
        # Normalises the residual stream before the lm_head projection.
        # Without this, activation scales can drift unpredictably across layers.
        self.norm = nn.LayerNorm(d_model)

        # -- Language model head ------------------------------------------
        # Linear projection from d_model -> vocab_size (no bias).
        # Output logits are fed to softmax / cross-entropy during training.
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # -- Weight tying -------------------------------------------------
        # Share the embedding matrix with the lm_head output matrix.
        # Same number of trainable parameters, better generalisation.
        self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Run the full model.

        Parameters
        ----------
        input_ids : (B, L)  -- integer token indices in [0, vocab_size)

        Returns
        -------
        logits : (B, L, vocab_size)
            Unnormalised log-probabilities for the *next* token at each position.
            Position i predicts token i+1 (used with targets[:, 1:] in training).
        """
        x = self.embedding(input_ids)        # (B, L, d_model)

        for layer in self.layers:
            x = layer(x)                     # (B, L, d_model) -- shape preserved

        x = self.norm(x)
        return self.lm_head(x)               # (B, L, vocab_size)

    def count_parameters(self) -> int:
        """Return the total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


---
## 6. Demos

The next three cells instantiate each component and confirm shapes and
parameter counts.

### What to look for

* **Shape preservation** — the output shape always matches the input shape
  `(B, L, d_model)`.  This is a deliberate design constraint: every block is
  a “same-shape” transformation that adds information to the residual stream
  without changing its dimensions, making stacking trivial.

* **Parameter count growth** — notice how many more parameters `MambaBlock`
  has compared to the vanilla `SSM` for the same `d_model`.  The extra
  parameters come from `in_proj`, `conv1d`, `x_proj`, `dt_proj`, and
  `out_proj` — the machinery needed to produce the input-dependent
  $\Delta$, $B$, $C$ that give Mamba its selectivity.

### 6a. Vanilla SSM


In [ ]:
torch.manual_seed(42)

ssm = SSM(d_model=32, d_state=16)
x = torch.randn(4, 64, 32)        # batch=4, seq_len=64, d_model=32
y = ssm(x)

print(f"Input  shape : {x.shape}")
print(f"Output shape : {y.shape}")
print(f"Parameters   : {sum(p.numel() for p in ssm.parameters()):,}")


**What to notice:**
* Output shape equals input shape — the SSM is a same-shape transformation.
* The parameter count is tiny: `A_log` + `B` + `C` + `D` + `log_dt` = four
  arrays of size `(d_model, d_state)` or `(d_model,)` → about 2 k parameters
  total.  This is the *simplest possible* learnable recurrence.

### 6b. Single Mamba block


In [ ]:
torch.manual_seed(42)

block = MambaBlock(d_model=64, d_state=16, d_conv=4, expand=2)
x = torch.randn(2, 128, 64)        # batch=2, seq_len=128, d_model=64
y = block(x)

print(f"Input  shape : {x.shape}")
print(f"Output shape : {y.shape}")
print(f"Parameters   : {sum(p.numel() for p in block.parameters()):,}")


**What to notice:**
* The shape is still `(B, L, d_model)` — adding gating, conv, and the
  selective scan changes *what* the transformation does, not its interface.
* Parameter count is ~20× larger than the vanilla SSM for the same `d_model`.
  The bulk comes from `in_proj` (d_model → 2·d_inner) and `out_proj`
  (d_inner → d_model), plus the two projections inside `SelectiveSSM`.

### 6c. Full Mamba language model


In [ ]:
torch.manual_seed(42)

model = MambaModel(
    vocab_size=256,
    d_model=128,
    n_layers=4,
    d_state=16,
    d_conv=4,
    expand=2,
)
token_ids = torch.randint(0, 256, (2, 64))   # batch=2, seq_len=64
logits = model(token_ids)

print(f"Token IDs shape : {token_ids.shape}")
print(f"Logits shape    : {logits.shape}")
print(f"Parameters      : {model.count_parameters():,}")


**What to notice:**
* Input is integer token IDs; output is `(B, L, vocab_size)` logits.  The
  model assigns a score to each vocabulary token at every position — the score
  for position `i` predicts what token should appear at `i+1`.
* With `n_layers=4` and `d_model=128` the model has ~500 k parameters — tiny
  by modern standards, but fully functional.
* `count_parameters()` is less than `vocab_size × d_model × 2` because weight
  tying means the embedding and lm_head share one matrix.


---
## 7. Mini Training Loop

### Causal language modelling

The standard training objective for a language model is **next-token prediction**:
given tokens $x_1, x_2, \ldots, x_{L-1}$, predict $x_2, x_3, \ldots, x_L$.

In practice:
1. Feed `input_ids = tokens[:, :-1]` (drop the last token) to the model.
2. The model produces logits of shape `(B, L-1, vocab_size)`.
3. Compare with `targets = tokens[:, 1:]` (drop the first token) using
   **cross-entropy loss**.

Because the Mamba blocks are causal (Conv1d left-padding + SSM recurrence
never looks ahead), position $i$ in the logits only depends on positions
$0 \ldots i$ of the input — exactly what we need for a valid language model.

### AdamW optimiser

`AdamW` (Adam with decoupled weight decay) is the de-facto standard for
training Transformers and SSMs.  Key hyper-parameters:
* `lr=1e-3` — a typical starting learning rate for small models.
* Default weight decay (`0.01`) provides mild regularisation.

### Reading the loss

Cross-entropy for a **uniform random baseline** over a vocabulary of size $V$
is $\ln V$.  Here $V = 64$, so the expected initial loss is
$\ln 64 \approx 4.16$.  Values above that indicate the model hasn’t learned
anything yet; decreasing values indicate it is picking up patterns.


In [ ]:
torch.manual_seed(0)

# Small model for a fast demo
model     = MambaModel(vocab_size=64, d_model=64, n_layers=2, d_state=8)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print(f"Random-baseline loss (ln 64): {math.log(64):.4f}")
print()

for step in range(5):
    # Random token sequences -- in a real task this would be actual text
    tokens = torch.randint(0, 64, (4, 32))       # (batch=4, seq_len=32)

    # Shift: inputs are tokens[0..L-2], targets are tokens[1..L-1]
    inputs, targets = tokens[:, :-1], tokens[:, 1:]

    # Forward pass: (4, 31, 64) logits
    logits = model(inputs)

    # Cross-entropy: flatten batch and sequence dimensions, then compare
    loss = F.cross_entropy(
        logits.reshape(-1, 64),   # (4*31, 64)
        targets.reshape(-1),      # (4*31,)
    )

    # Backward pass and parameter update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"step {step + 1}: loss = {loss.item():.4f}")

print()
print("Done \u2713")


**What to notice:**
* The initial loss is close to the random baseline ($\approx \ln 64 \approx 4.16$),
  as expected for randomly initialised weights on random data.
* Loss fluctuates between steps because the training data itself is random —
  there is no real pattern to learn.  On actual text data you would see a
  consistent downward trend over many more steps.
* Even on toy data, `loss.backward()` completes without errors, confirming
  that gradients flow through the entire computation graph:
  `lm_head → norm → MambaBlock (×2) → embedding`.
* The `optimizer.step()` call updates every parameter simultaneously:
  embeddings, layer norms, projection matrices, SSM matrices, and conv filters.

---

### Recap: how the pieces fit together

| Section | Module | Role in the full model |
|---------|--------|------------------------|
| 2 | `SSM` | Fixed-parameter recurrence; the simplest possible sequence layer |
| 3 | `SelectiveSSM` | Input-dependent recurrence; the “engine” that gives Mamba selectivity |
| 4 | `MambaBlock` | Wraps `SelectiveSSM` with gating, conv, norm, and residual; the stacking unit |
| 5 | `MambaModel` | Stacks $N$ `MambaBlock`s between an embedding and an LM head |

### Next steps

* **Scale up**: increase `n_layers`, `d_model`, and train on real text (e.g. a
  character-level corpus or a tokenised dataset from Hugging Face).
* **Efficiency**: replace the Python `for t in range(L)` scan in `SelectiveSSM`
  with a fused CUDA kernel (the `mamba_ssm` package) for 10–100× speedup.
* **Explore selectivity**: inspect the `dt` values from `SelectiveSSM` to see
  which positions get large vs. small time steps — the model should learn to
  “focus” on informative tokens by assigning them larger $\Delta$.
